# Serve large models on SageMaker with DeepSpeed Container


---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook. 

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/generative_ai|sm-djl_deepspeed_bloom_176b_deploy.ipynb)

---


In this notebook, we explore how to host a large language model on SageMaker using the latest container launched using DeepSpeed and DJL. DJL provides for the serving framework while DeepSpeed is the key sharding library we leverage to enable hosting of large models. We use DJLServing as the model serving solution in this example. DJLServing is a high-performance universal model serving solution powered by the Deep Java Library (DJL) that is programming language agnostic. To learn more about DJL and DJLServing, you can refer to our recent blog post (https://aws.amazon.com/blogs/machine-learning/deploy-large-models-on-amazon-sagemaker-using-djlserving-and-deepspeed-model-parallel-inference/).

Language models have recently exploded in both size and popularity. In 2018, BERT-large entered the scene and, with its 340M parameters and novel transformer architecture, set the standard on NLP task accuracy. Within just a few years, state-of-the-art NLP model size has grown by more than 500x with models such as OpenAI’s 175 billion parameter GPT-3 and similarly sized open source Bloom 176B raising the bar on NLP accuracy. This increase in the number of parameters is driven by the simple and empirically-demonstrated positive relationship between model size and accuracy: more is better. With easy access from models zoos such as Hugging Face and improved accuracy in NLP tasks such as classification and text generation, practitioners are increasingly reaching for these large models. However, deploying them can be a challenge because of their size.

Model parallelism can help deploy large models that would normally be too large for a single GPU. With model parallelism, we partition and distribute a model across multiple GPUs. Each GPU holds a different part of the model, resolving the memory capacity issue for the largest deep learning models with billions of parameters. This notebook uses tensor parallelism techniques which allow GPUs to work simultaneously on the same layer of a model and achieve low latency inference relative to a pipeline parallel solution.

SageMaker has rolled out DeepSpeed container which now provides users with the ability to leverage the managed serving capabilities and help to provide the un-differentiated heavy lifting.

In this notebook, we deploy the open source Bloom 176B quantized model across GPU's on a ml.p4d.24xlarge instance. DeepSpeed is used for tensor parallelism inference while DJLServing handles inference requests and the distributed workers. For further reading on DeepSpeed you can refer to https://arxiv.org/pdf/2207.00032.pdf 


## License agreement
View license information https://huggingface.co/spaces/bigscience/license for this model including the use-based restrictions in Section 5 before using the model. 


In [1]:
# [E2E exec copy] pip install disabled: the .v3test-venv already has the local
# V3 source installed. (Source -v3 notebook keeps the real pip install.)
# %pip install -Uqq boto3 awscli 'sagemaker>=3.0'

## Optional Section to Download Model from Hugging Face Hub

Use this section of you are interested in downloading the model directly from Huggingface hub and storing in your own S3 bucket. Please change the variable "install_model_locally" to True in that case.

**However, this notebook currently leverages the model stored in AWS public S3 location for ease of use. So you can skip this step**

The below step to download and then upload to S3 can take several minutes since the model size is extremely large

In [2]:
install_model_locally = False

In [3]:
if install_model_locally:
    %pip install huggingface-hub -Uqq

In [4]:
if install_model_locally:

    from huggingface_hub import snapshot_download
    from pathlib import Path

    # - This will download the model into the ./model directory where ever the jupyter file is running
    local_model_path = Path("./model")
    local_model_path.mkdir(exist_ok=True)
    model_name = "microsoft/bloom-deepspeed-inference-int8"
    commit_hash = "aa00a6626f6484a2eef68e06d1e089e4e32aa571"

    # - Leverage the snapshot library to donload the model since the model is stored in repository using LFS
    snapshot_download(repo_id=model_name, revision=commit_hash, cache_dir=local_model_path)

    # - Upload to S3 using AWS CLI
    s3_model_prefix = "hf-large-model-djl-ds/model"  # folder where model checkpoint will go
    model_snapshot_path = list(local_model_path.glob("**/snapshots/*"))[0]

    !aws s3 cp --recursive {model_snapshot_path} s3://{bucket}/{s3_model_prefix}

## Create SageMaker compatible Model artifact and Upload Model to S3

SageMaker needs the model to be in a Tarball format. 

The tarball is in the following format

```
code
├──── 
│   └── model.py
│   └── serving.properties

``` 

- `model.py` is the key file which will handle any requests for serving. It is also responsible for loading the model from S3
- `serving.properties` is the configuration file that can be used to configure the model server.


In [5]:
# v3: `import sagemaker` + `from sagemaker import image_uris` ->
# sagemaker-core session helpers + `from sagemaker.core import image_uris`.
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session, get_execution_role
import boto3
import os
import time
import json
from pathlib import Path

/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_data_url" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_data_source" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_description" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/

[07/16/26 15:58:41] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=1386060;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=1386061;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_version" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


#### Create required variables and initialize them to create the endpoint, we leverage boto3 for this

In [6]:
# [E2E exec copy] Pin region to us-west-2 (us-west-1 has no g5) and inject an
# explicit execution role, since get_execution_role() only works inside SageMaker.
# Source -v3 keeps role=get_execution_role() / Session().
_boto_sess = boto3.Session(region_name="us-west-2")
role = "arn:aws:iam::729646638167:role/service-role/AmazonSageMaker-ExecutionRole-20251201T194045"
sess = Session(boto_session=_boto_sess)  # sagemaker-core session
bucket = sess.default_bucket()  # bucket to house artifacts
default_bucket_prefix = sess.default_bucket_prefix
model_bucket = f"sagemaker-example-files-prod-{sess.boto_region_name}"
s3_code_prefix = "hf-large-model-djl-ds/code"  # folder within bucket where code artifact will go
s3_model_prefix = "models/bloom-176B/raw_model_microsoft/"  # "bloom-176B/raw_model_microsoft/"  # folder where model checkpoint will go
# S3 URI--  s3://sagemaker-example-files-prod-{region}/models/bloom-176B/raw_model_microsoft/ -

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    s3_model_prefix = f"{default_bucket_prefix}/{s3_model_prefix}"
    s3_code_prefix = f"{default_bucket_prefix}/{s3_code_prefix}"

region = sess.boto_region_name
account_id = sess.account_id()

s3_client = boto3.client("s3")
sm_client = boto3.client("sagemaker")
smr_client = boto3.client("sagemaker-runtime")

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=1386066;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=1386067;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

[07/16/26 15:58:42] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=1386072;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=1386073;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

**Image URI of the DJL Container to be used**

In [7]:
inference_image_uri = image_uris.retrieve(
    framework="djl-deepspeed", region=sess.boto_session.region_name, version="0.21.0"
)
print(f"Image going to be used is ---- > {inference_image_uri}")

                    INFO     Ignoring unnecessary instance type: None.                            ]8;id=1386080;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=1386081;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py#535\535]8;;\

Image going to be used is ---- > 763104351884.dkr.ecr.us-west-2.amazonaws.com/djl-inference:0.21.0-deepspeed0.8.3-cu117


**Create the Tarball and then upload to S3 location**

In [8]:
!mkdir -p code_bloom176

In [9]:
%%writefile code_bloom176/model.py
# [E2E exec copy] Lightweight DJL Python-engine stand-in handler.
# The source -v3 notebook keeps the real BLOOM-176B DeepSpeed handler; here we
# swap in a tiny echo handler so the endpoint comes up in seconds on a small GPU
# (no DeepSpeed init, no 176B download). Every migrated V3 SDK call
# (Model/EndpointConfig/Endpoint.create, wait_for_status, Endpoint.invoke,
# delete) still runs verbatim against a real SageMaker endpoint.
from djl_python import Input, Output
import logging


def handle(inputs: Input):
    """
    inputs: Contains the configurations from serving.properties
    """
    if inputs.is_empty():
        # Model server makes an empty call to warmup the model on startup
        return None

    data = inputs.get_as_json()
    input_sentences = data.get("inputs", [])
    if isinstance(input_sentences, str):
        input_sentences = [input_sentences]
    params = data.get("parameters", {})
    logging.info(f"stand-in handler received {len(input_sentences)} inputs, params={params}")

    # Echo the prompt back with a deterministic completion so the response shape
    # matches the real handler ({"outputs": [...]}).
    outputs = [f"{s} [stand-in completion]" for s in input_sentences]
    result = {"outputs": outputs}
    return Output().add_as_json(result)

Overwriting code_bloom176/model.py


#### Serving.properties has engine parameter which tells the DJL model server to use the DeepSpeed engine to load the model

Here is a list of settings that we use in this configuration file -

- `engine`: The engine for DJL to use. In this case, we intend to use Accelerate and hence set it to Python.
- `option.entryPoint`: The entrypoint python file or module. This should align with the engine that is being used.
- `option.s3url`: Set this to the URI of the Amazon S3 bucket that contains the model. When this is set, the container leverages s5cmd to download the model from s3. This is extremely fast and useful when downloading large models like this one.

The container downloads the model into the /tmp space on the container because SageMaker maps the /tmp to the Amazon Elastic Block Store (Amazon EBS) volume that is mounted when we specify the endpoint creation parameter VolumeSizeInGB. It leverages s5cmd(https://github.com/peak/s5cmd) which offers a very fast download speed and hence extremely useful when downloading large models.

For instances like p4dn, which come pre-built with the volume instance, we can continue to leverage the /tmp on the container. The size of this mount is large enough to hold the model.

For more details on the configuration options and an exhaustive list, you can refer the documentation - https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints-large-model-configuration.html

In [10]:
# [E2E exec copy] Use the DJL Python engine with the tiny stand-in handler
# instead of the DeepSpeed engine + 176B s3url. Source -v3 keeps engine=DeepSpeed.
props = """
engine = Python
"""
print(props, file=open("code_bloom176/serving.properties", "a"))

In [11]:
!rm model.tar.gz
!tar czvf model.tar.gz code_bloom176

a code_bloom176
a code_bloom176/serving.properties
a code_bloom176/model.py


In [12]:
s3_code_artifact = sess.upload_data("model.tar.gz", bucket, s3_code_prefix)
print(f"S3 Code or Model tar ball uploaded to --- > {s3_code_artifact}")

S3 Code or Model tar ball uploaded to --- > s3://sagemaker-us-west-2-729646638167/hf-large-model-djl-ds/code/model.tar.gz


In [13]:
print(f"S3 Model Prefix where the model files are -- > {s3_model_prefix}")
print(f"S3 Model Bucket is -- > {model_bucket}")

S3 Model Prefix where the model files are -- > models/bloom-176B/raw_model_microsoft/
S3 Model Bucket is -- > sagemaker-example-files-prod-us-west-2


### This is optional in case you want to use VpcConfig to specify when creating the end points

For more details you can refer to this link https://docs.aws.amazon.com/sagemaker/latest/dg/host-vpc.html

The below is just an example to extract information about Security Groups and Subnets needed to configure

In [14]:
!aws ec2 describe-security-groups --filter Name=vpc-id,Values=<use vpcId> | python3 -c "import sys, json; print(json.load(sys.stdin)['SecurityGroups'])"

zsh:1: parse error near `|'


In [15]:
# - provide networking configs if needed.
security_group_ids = []  # add the security group id's
subnets = []  # add the subnet id for this vpc
privateVpcConfig = {"SecurityGroupIds": security_group_ids, "Subnets": subnets}
print(privateVpcConfig)

{'SecurityGroupIds': [], 'Subnets': []}


### To create the end point the steps are:

1. Create the Model using the Image container and the Model Tarball uploaded earlier
2. Create the endpoint config using the following key parameters

    a) Instance Type is ml.p4d.24xlarge 
    
    b) ModelDataDownloadTimeoutInSeconds is 2400 which is needed to ensure the Model downloads from S3 successfully,
    
    c) ContainerStartupHealthCheckTimeoutInSeconds is 2400 to ensure health check starts after the model is ready
    
3. Create the end point using the endpoint config created    
    

In [16]:
# v3: `sm_client.create_model(PrimaryContainer={...})` ->
# `Model.create(primary_container=ContainerDefinition(...))` from sagemaker-core.
# `sagemaker.utils.name_from_base` -> `sagemaker.core.common_utils.name_from_base`.
from sagemaker.core.resources import Model
from sagemaker.core.shapes import ContainerDefinition
from sagemaker.core.common_utils import name_from_base

model_name = name_from_base(f"bloom-djl-ds")
print(model_name)

model = Model.create(
    model_name=model_name,
    execution_role_arn=role,
    primary_container=ContainerDefinition(
        image=inference_image_uri,
        model_data_url=s3_code_artifact,
    ),
    # Uncomment if providing networking configs
    # vpc_config=privateVpcConfig,
)
model_arn = model.model_arn

print(f"Created Model: {model_arn}")

bloom-djl-ds-2026-07-16-22-58-44-231
sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


[07/16/26 15:58:44] INFO     Creating model resource.                                            ]8;id=1386088;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1386089;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#20593\20593]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=1386096;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=1386097;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py#361\361]8;;\

                    INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=1386103;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=1386104;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py#375\375]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=1386109;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=1386110;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=1386115;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=1386116;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

Created Model: arn:aws:sagemaker:us-west-2:729646638167:model/bloom-djl-ds-2026-07-16-22-58-44-231


VolumnSizeInGB has been commented out. You should use this value for Instance types which support EBS volume mounts. The current instance we are using comes with a pre-configured space and does not support additional volume mounts

In [17]:
# v3: `sm_client.create_endpoint_config(ProductionVariants=[{...}])` ->
# `EndpointConfig.create(production_variants=[ProductionVariant(...)])`.
from sagemaker.core.resources import EndpointConfig
from sagemaker.core.shapes import ProductionVariant

endpoint_config_name = f"{model_name}-config"
endpoint_name = f"{model_name}-endpoint"

endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    production_variants=[
        ProductionVariant(
            variant_name="variant1",
            model_name=model_name,
            # [E2E exec copy] ml.p4d.24xlarge -> ml.g5.xlarge (source -v3 keeps p4d).
            instance_type="ml.g5.xlarge",
            initial_instance_count=1,
            # volume_size_in_gb=400,
            model_data_download_timeout_in_seconds=2400,
            container_startup_health_check_timeout_in_seconds=2400,
        ),
    ],
)
endpoint_config

[07/16/26 15:58:45] INFO     Creating endpoint_config resource.                                  ]8;id=1386122;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1386123;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#11069\11069]8;;\

EndpointConfig(endpoint_config_name='bloom-djl-ds-2026-07-16-22-58-44-231-config', endpoint_config_arn='arn:aws:sagemaker:us-west-2:729646638167:endpoint-config/bloom-djl-ds-2026-07-16-22-58-44-231-config', production_variants=[ProductionVariant(variant_name='variant1', model_name='bloom-djl-ds-2026-07-16-22-58-44-231', initial_instance_count=1, instance_type='ml.g5.xlarge', instance_pools=Unassigned(), variant_instance_provision_timeout_in_seconds=Unassigned(), initial_variant_weight=1.0, accelerator_type=Unassigned(), core_dump_config=Unassigned(), serverless_config=Unassigned(), volume_size_in_gb=Unassigned(), model_data_download_timeout_in_seconds=2400, container_startup_health_check_timeout_in_seconds=2400, enable_ssm_access=Unassigned(), managed_instance_scaling=Unassigned(), routing_config=Unassigned(), inference_ami_version=Unassigned(), capacity_reservation_config=Unassigned())], data_capture_config=Unassigned(), kms_key_id=Unassigned(), creation_time=datetime.datetime(2026, 7

In [18]:
# v3: `sm_client.create_endpoint(...)` -> `Endpoint.create(...)`.
from sagemaker.core.resources import Endpoint

endpoint = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_config_name,
)
print(f"Created Endpoint: {endpoint.endpoint_arn}")

                    INFO     Creating endpoint resource.                                         ]8;id=1386129;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1386130;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10228\10228]8;;\

Created Endpoint: arn:aws:sagemaker:us-west-2:729646638167:endpoint/bloom-djl-ds-2026-07-16-22-58-44-231-endpoint


#### Wait for the end point to be created. This can take a few minutes. Please be patient
However, while that happens, let us look at the critical areas of the helper files we are using to load the model
1. We will look at the code snippets for model.py to see the model downloading mechanism
2. Serving.properties to see the environment related properties

In [19]:
# This is the code snippet which is responsible to load the model from S3
! sed -n '40,60p' code_bloom176/model.py

In [20]:
# This is the code snippet which shows the environment variables being used to customize runtime
! sed -n '1,3p' code_bloom176/serving.properties


engine = Python



In [21]:
# v3: manual `describe_endpoint` polling loop -> `endpoint.wait_for_status("InService")`.
endpoint.wait_for_status("InService")
endpoint.refresh()

print("Arn: " + endpoint.endpoint_arn)
print("Status: " + endpoint.endpoint_status)

Output()

[07/16/26 16:03:06] INFO     Final Resource Status: InService                                    ]8;id=1386136;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1386137;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10484\10484]8;;\

Arn: arn:aws:sagemaker:us-west-2:729646638167:endpoint/bloom-djl-ds-2026-07-16-22-58-44-231-endpoint
Status: InService


#### Leverage the Boto3 api to invoke the endpoint. 

This is a generative model, so we pass in a Text (specified in the 'input' field in the json) as a prompt and Model will complete the sentence and return the results. More details on these parameters can be found at https://huggingface.co/docs/api-inference/detailed_parameters#text-generation-task. Some quick explainations are below
1. temperature -- > The temperature of the sampling operation. 1 means regular sampling, 0 means always take the highest score and 100 means uniform probability
2. max_new_tokens -- > The amount of new tokens or text to be gnerated. More tokens will increase the prediction time
3. num_beams -- > Beam Search keeps track of the n-th most likely word sequences.


In [22]:
%%time
# v3: `smr_client.invoke_endpoint(...)["Body"].read()` ->
# `endpoint.invoke(body=..., content_type=...)` which returns an
# `InvokeEndpointOutput`; `.body` is a boto3 StreamingBody.
response = endpoint.invoke(
    body=json.dumps(
        {
            "inputs": ["Amazon.com is the best "],
            "parameters": {
                "min_length": 5,
                "max_new_tokens": 100,
                "temperature": 0.8,
                "num_beams": 5,
                "no_repeat_ngram_size": 2,
            },
        }
    ),
    content_type="application/json",
)
response.body.read().decode("utf8")

CPU times: user 19.8 ms, sys: 6.83 ms, total: 26.7 ms
Wall time: 321 ms


'{\n  "outputs":[\n    "Amazon.com is the best  [stand-in completion]"\n  ]\n}'

#### With do_sample to false we are making a greedy optimization for token generation

In [23]:
%%time
# -- Greedy generation
response = endpoint.invoke(
    body=json.dumps(
        {
            "inputs": ["Amazon.com is the best ", "Large Models are the way to go"],
            "parameters": {
                "min_length": 5,
                "max_new_tokens": 10,
                "do_sample": False,
                "early_stopping": True,
            },
            "padding": True,
        }
    ),
    content_type="application/json",
)
response.body.read().decode("utf8")

CPU times: user 2.23 ms, sys: 339 μs, total: 2.57 ms
Wall time: 37.1 ms


'{\n  "outputs":[\n    "Amazon.com is the best  [stand-in completion]",\n    "Large Models are the way to go [stand-in completion]"\n  ]\n}'

## Conclusion
In this post, we demonstrated how to use SageMaker large model inference containers to host two large language models, BLOOM-176B and OPT-30B. We used DeepSpeed’s model parallel techniques with multiple GPUs on a single SageMaker machine learning instance. For more details about Amazon SageMaker and its large model inference capabilities, refer to the following:

* Amazon SageMaker now supports deploying large models through configurable volume size and timeout quotas (https://aws.amazon.com/about-aws/whats-new/2022/09/amazon-sagemaker-deploying-large-models-volume-size-timeout-quotas/)
* Real-time inference – Amazon SageMaker (https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints.html)


## Clean Up

In [24]:
# - Delete the end point
# v3: `sm_client.delete_endpoint(...)` -> `endpoint.delete()`.
endpoint.delete()

[07/16/26 16:03:08] INFO     Deleting Endpoint - bloom-djl-ds-2026-07-16-22-58-44-231-endpoint   ]8;id=1392259;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1392260;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10428\10428]8;;\

In [25]:
# - In case the end point failed we still want to delete the model
# v3: `sm_client.delete_endpoint_config(...)` / `delete_model(...)` ->
# `endpoint_config.delete()` / `model.delete()`.
endpoint_config.delete()
model.delete()

                    INFO     Deleting EndpointConfig -                                           ]8;id=1392266;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1392267;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#11220\11220]8;;\
                             bloom-djl-ds-2026-07-16-22-58-44-231-config                                           

[07/16/26 16:03:09] INFO     Deleting Model - bloom-djl-ds-2026-07-16-22-58-44-231               ]8;id=1392273;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1392274;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#20740\20740]8;;\

#### Optionally delete the model checkpoint from S3

In [26]:
!aws s3 rm --recursive s3://{bucket}/{s3_model_prefix}

In [27]:
s3_client = boto3.client("s3")

In [28]:
# Use .get("Contents", []) because list_objects omits the "Contents" key when
# no objects match the prefix (e.g. when install_model_locally=False, so no
# model checkpoint was ever uploaded under this prefix).
len(s3_client.list_objects(Bucket=bucket, Prefix=f"{s3_model_prefix}/").get("Contents", []))

0

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/generative_ai|sm-djl_deepspeed_bloom_176b_deploy.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/generative_ai|sm-djl_deepspeed_bloom_176b_deploy.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/generative_ai|sm-djl_deepspeed_bloom_176b_deploy.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/generative_ai|sm-djl_deepspeed_bloom_176b_deploy.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/generative_ai|sm-djl_deepspeed_bloom_176b_deploy.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/generative_ai|sm-djl_deepspeed_bloom_176b_deploy.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/generative_ai|sm-djl_deepspeed_bloom_176b_deploy.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/generative_ai|sm-djl_deepspeed_bloom_176b_deploy.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/generative_ai|sm-djl_deepspeed_bloom_176b_deploy.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/generative_ai|sm-djl_deepspeed_bloom_176b_deploy.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/generative_ai|sm-djl_deepspeed_bloom_176b_deploy.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/generative_ai|sm-djl_deepspeed_bloom_176b_deploy.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/generative_ai|sm-djl_deepspeed_bloom_176b_deploy.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/generative_ai|sm-djl_deepspeed_bloom_176b_deploy.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/generative_ai|sm-djl_deepspeed_bloom_176b_deploy.ipynb)
